In [12]:
using LinearAlgebra: norm, dot

function gdoptimize(f, g!, fg!, x0::AbstractArray{T}, linesearch,
                    maxiter::Int = 10000,
                    g_rtol::T = sqrt(eps(T)), g_atol::T = eps(T)) where T <: Number
    x = copy(x0)
    gvec = similar(x)
    g!(gvec, x)
    fx = f(x)

    gnorm = norm(gvec)
    gtol = max(g_rtol*gnorm, g_atol)

    # Univariate line search functions
    ϕ(α) = f(x .+ α.*s)
    function dϕ(α)
        g!(gvec, x .+ α.*s)
        return dot(gvec, s)
    end
    function ϕdϕ(α)
        phi = fg!(gvec, x .+ α.*s)
        dphi = dot(gvec, s)
        return (phi, dphi)
    end

    s = similar(gvec) # Step direction

    iter = 0
    while iter < maxiter && gnorm > gtol
        iter += 1
        s .= -gvec

        dϕ_0 = dot(s, gvec)
        
        println([ϕ, dϕ, ϕdϕ, 1.0, fx, dϕ_0])
        
        α, fx = linesearch(ϕ, dϕ, ϕdϕ, 1.0, fx, dϕ_0)

        @. x = x + α*s
        g!(gvec, x)
        gnorm = norm(gvec)
    end

    return (fx, x, iter)
end

f(x) = (1.0 - x[1])^2 + 100.0 * (x[2] - x[1]^2)^2

function g!(gvec, x)
    gvec[1] = -2.0 * (1.0 - x[1]) - 400.0 * (x[2] - x[1]^2) * x[1]
    gvec[2] = 200.0 * (x[2] - x[1]^2)
    gvec
end

function fg!(gvec, x)
    g!(gvec, x)
    f(x)
end

fg! (generic function with 1 method)

In [14]:
x0 = [-1.5, 1.0]

using LineSearches
ls = BackTracking(order=3)
fx_bt3, x_bt3, iter_bt3 = gdoptimize(f, g!, fg!, x0, ls, 1)

Any[var"#ϕ#13"{typeof(f), Vector{Float64}}(Main.f, Core.Box([755.0, 250.0]), [-1.5, 1.0]), var"#dϕ#14"{typeof(g!), Vector{Float64}, Vector{Float64}}(Main.g!, Core.Box([755.0, 250.0]), [-755.0, -250.0], [-1.5, 1.0]), var"#ϕdϕ#15"{typeof(fg!), Vector{Float64}, Vector{Float64}}(Main.fg!, Core.Box([755.0, 250.0]), [-755.0, -250.0], [-1.5, 1.0]), 1.0, 162.5, -632525.0]


(148.6934342466702, [0.7178555030477503, 1.7343892394197846], 1)

In [5]:
ls = StrongWolfe()
fx_sw, x_sw, iter_sw = gdoptimize(f, g!, fg!, x0, ls)

(1.3851361590406795e-10, [1.0000117640032875, 1.0000235630509198], 9649)

In [50]:
a0=2.1
phi(a) = (a-a0)^2

function dphi(a)
        return 2.0 * (a-a0)
    end

function phidphi(a)
    return (phi(a), dphi(a))
end

ls=StrongWolfe(c_1 = 1e-4, c_2 = 0.9, ρ = 2.0)
x1, f1 = ls(phi, dphi, phidphi, 99.0, phi(0), dphi(0))

(2.0999999999999943, 3.332937324558775e-29)